# Band-split層

Band-splitは、「Band-split RNN[^Luo22]」で初めて提案された、音楽音源分離モデルのフロントエンド構造です。

Band-split RNNは発表当時から高い分離性能を示していましたが、このアイデアをTransformer系モデルに応用した「BS-Roformer[^bsr]」「Mel-Band Roformer[^mbr]」が新たなSOTAを達成したことにより、音源分離分野で注目が集まりつつあります。

Band-split層の仕組み自体は単純です。入力されたSTFT振幅スペクトログラムを周波数軸でSub-bandに分割し、それぞれ別々のLinear層に入力するだけです。BS-RNNおよびBS-Roformerは、Band-split層から出力されたテンソルを時間方向・周波数方向に時系列モデルに入力することで、強力なモデルを構しています。

Sub-bandの切り方は音源分離の性能を左右するとされています。Band-split RNN論文が示した比較実験によると、①低域を狭く、高域を広く切り出す（1kHzあたりが境界線）、②できるだけ多くのSub-bandを切り出すと、性能が高くなる傾向があるようです。

Mel-band RoFormerのように、各Sub-bandの帯域を一部重畳させることでSub-band数を稼ぐのも一つの選択肢です。

[^Luo22]: Y. Luo and J. Yu, "Music Source Separation With Band-Split RNN," in IEEE/ACM Transactions on Audio, Speech, and Language Processing, vol. 31, pp. 1893-1901, 2023. URL: https://ieeexplore.ieee.org/document/10121418
[^bsr]: W. T. Lu et al., Music Source Separation with Band-Split RoPE Transformer, in EEE International Conference on Acoustics, Speech and Signal Processing (ICASSP), 2023. URL: https://ieeexplore.ieee.org/document/10446843
[^mbr]: J.C Wang et al., Mel-Band RoFormer for Music Source Separation, ArXiv, abs/2310.01809. https://arxiv.org/abs/2310.01809

In [ ]:
import torch
import torch.nn as nn

"""
Source code from:  https://github.com/lucidrains/BS-RoFormer/
"""

# BS-Roformerのデフォルト分割方式
DEFAULT_FREQS_PER_BANDS = (
  2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
  2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
  2, 2, 2, 2,
  4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
  12, 12, 12, 12, 12, 12, 12, 12,
  24, 24, 24, 24, 24, 24, 24, 24,
  48, 48, 48, 48, 48, 48, 48, 48,
  128, 129,
)

class BandSplit(nn.Module):
    def __init__(self, dim: int, dim_inputs: tuple[int, ...]):
        super().__init__()
        self.dim_inputs = dim_inputs
        self.to_features = nn.ModuleList([])

        for dim_in in dim_inputs:
            net = nn.Sequential(
                nn.LayerNorm(dim_in),
                nn.Linear(dim_in, dim)
            )
            self.to_features.append(net)

    def forward(self, x):
        x = x.split(self.dim_inputs, dim=-1)        # 入力を周波数バンドに分割

        outs = []
        for split_input, to_feature in zip(x, self.to_features):
            outs.append(to_feature(split_input))    # 各周波数バンドを特徴量に変換
        
        return torch.stack(outs, dim=-2)

x_in = torch.randn(4, 5000, 1025)       # (batch, seq_len, n_freq_bins)
band_split = BandSplit(256, DEFAULT_FREQS_PER_BANDS)
x_out = band_split(x_in)
print("Shape of x_out:", x_out.shape)   # (batch, seq_len, n_subbands, latent_dim)

Shape of x_out: torch.Size([4, 5000, 62, 256])
